"An LLM gateway sits between our application and model providers. It can route requests across multiple providers based on rate limits, availability, latency, cost, and model capability. This prevents our application from depending entirely on a single provider and allows failover when a provider is unavailable or rate-limited."

litellm we will use as a gateway

In [ ]:
from litellm import completion

response = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

In [ ]:
from litellm import completion

response = completion(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "Explain Redis"}
    ],
    fallbacks=[
        {"model": "claude-3-5-sonnet"},
        {"model": "gemini/gemini-2.0-flash"}
    ]
)

In [ ]:
from litellm import completion

response = completion(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "Explain Redis"}
    ],
    fallbacks=[
        {"model": "claude-3-5-sonnet"},
        {"model": "gemini/gemini-2.0-flash"}
    ]
)

. from litellm.caching import Cache
from litellm.caching import Cache

LiteLLM ke andar jo Cache functionality/class hai, usko import kar rahe ho.

Matlab:

"Mujhe LiteLLM ka cache mechanism use karna hai."

2. litellm.cache = Cache(...)
litellm.cache = Cache(...)

Yahan tum LiteLLM ko bol rahe ho:

"Jab bhi caching use karni ho, is Cache configuration ko use karna."

In [ ]:
from litellm import completion
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(
    type="redis",
    host="localhost",
    port=6379,
    password="..."
)

response = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Redis?"}
    ],
    caching=True
)
cost=completion_cost(completion_response=response)

In [ ]:
from litellm import Router

router = Router(
    model_list=[
        {
            "model_name": "gpt-4o",
            "litellm_params": {
                "model": "openai/gpt-4o",
                "api_key": "OPENAI_API_KEY"
            }
        },
        {
            "model_name": "gpt-4o",
            "litellm_params": {
                "model": "openai/gpt-4o",
                "api_key": "OPENAI_API_KEY"
            }
        },
        {
            "model_name": "claude",
            "litellm_params": {
                "model": "anthropic/claude-3-5-sonnet",
                "api_key": "ANTHROPIC_API_KEY"
            }
        }
    ],
    routing_strategy="simple-shuffle"
)

In [ ]:
router.completion(model="gpt-4o",  messages=[
        {"role": "user", "content": "What is Redis?"}
    ])

load balancing accross multiple api keys
routing_strategy="simple-shuffle"
routing_strategy="least-busy"
routing_strategy="latency-based-routing"



llm gate integration with langchain is here 

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate

llm = ChatLiteLLM(
    model="gpt-4o",
    temperature=0
)
fallback_1=ChatLiteLLM(
    model="gpt-4o-mini",
    temperature=0
)

fallback_2=ChatLiteLLM(
    model="anthropic/claude-3-5-sonnet",
    temperature=0
)

robust_llm=llm.with_fallbacks([fallback_1,fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

chain = prompt | robust_llm

response = chain.invoke({
    "question": "What is Kafka?"
})

print(response.content)

In [ ]:
import litellm
from litellm import completion


# 1. Map routing category -> model
MODEL_MAP = {
    "simple": "openai/gpt-4o-mini",
    "coding": "openai/gpt-4o",
    "reasoning": "openai/gpt-4o",
}


# 2. Routing prompt
ROUTER_PROMPT = """
Classify the user's query into exactly ONE category.

Categories:
- simple
- coding
- reasoning

Return ONLY the category name.
No explanation.

User query:
"""


def get_route(user_query: str) -> str:

    response = completion(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": ROUTER_PROMPT + user_query,
            }
        ],
        temperature=0,
    )

    route = response.choices[0].message.content.strip().lower()

    return route


def chatbot(user_query: str):

    # Step 1: Ask routing LLM
    route = get_route(user_query)

    print("Router selected:", route)

    # Step 2: Get model from map
    model = MODEL_MAP.get(route)

    if not model:
        # fallback route
        model = MODEL_MAP["simple"]

    # Step 3: Actual completion call
    response = completion(
        model=model,
        messages=[
            {
                "role": "user",
                "content": user_query,
            }
        ],
        temperature=0,
    )

    return response.choices[0].message.content


if __name__ == "__main__":

    query = input("You: ")

    answer = chatbot(query)

    print("Bot:", answer)

In [ ]:
import time
import litellm
from litellm import completion


# --------------------------------
# 1. MODEL ROUTING MAP
# --------------------------------

MODEL_MAP = {
    "simple": "openai/gpt-4o-mini",
    "coding": "openai/gpt-4o",
    "reasoning": "openai/gpt-4o",
}


# --------------------------------
# 2. INPUT CALLBACK
# --------------------------------

def input_guardrail(kwargs, *args, **extra):

    messages = kwargs.get("messages", [])

    if not messages:
        return

    user_message = messages[-1].get("content", "")

    print("\n[INPUT CALLBACK]")
    print("Input:", user_message)

    # Very simple example guardrail
    blocked_words = [
        "password",
        "secret_key",
        "api_key",
    ]

    for word in blocked_words:
        if word.lower() in user_message.lower():
            raise ValueError(
                "Request blocked by input guardrail"
            )


# --------------------------------
# 3. SUCCESS CALLBACK
# --------------------------------

def success_logger(kwargs, response_obj, start_time, end_time):

    latency = end_time - start_time

    print("\n[SUCCESS CALLBACK]")

    print("Model:", kwargs.get("model"))

    print("Latency:", round(latency, 3), "seconds")

    if hasattr(response_obj, "usage"):
        print(
            "Tokens:",
            response_obj.usage.total_tokens
        )


# Register callbacks
litellm.input_callback = [input_guardrail]
litellm.success_callback = [success_logger]


# --------------------------------
# 4. ROUTER LLM
# --------------------------------

def classify_query(user_query: str):

    response = completion(
        model="openai/gpt-4o-mini",

        messages=[
            {
                "role": "system",
                "content": """
Classify the user query into exactly ONE category.

Categories:

simple
coding
reasoning

Return ONLY one word.
No explanation.
"""
            },
            {
                "role": "user",
                "content": user_query
            }
        ],

        temperature=0
    )

    category = (
        response.choices[0]
        .message
        .content
        .strip()
        .lower()
    )

    return category


# --------------------------------
# 5. ACTUAL CHATBOT
# --------------------------------

def chatbot(user_query):

    # STEP 1
    category = classify_query(user_query)

    print("\nRouter selected:", category)

    # Safety fallback
    if category not in MODEL_MAP:
        category = "simple"

    # STEP 2
    selected_model = MODEL_MAP[category]

    print("Selected model:", selected_model)

    # STEP 3
    response = completion(
        model=selected_model,

        messages=[
            {
                "role": "user",
                "content": user_query
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


# --------------------------------
# 6. TEST
# --------------------------------

if __name__ == "__main__":

    query = input("You: ")

    try:

        answer = chatbot(query)

        print("\nBOT:")
        print(answer)

    except Exception as e:

        print("Request failed:", e)